# Usando pandas aplicado ao banco Northwind

**Exercícios - Fase 1**

**Antes de rodar:** siga a seção *Preparando o ambiente* do [README da fase](../README.md).

## Preparação

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
import northwind as nw

pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
produtos     = nw.tabela("products")
categorias   = nw.tabela("categories")
clientes     = nw.tabela("customers")
pedidos      = nw.tabela("orders")
itens        = nw.tabela("order_details")
funcionarios = nw.tabela("employees")
fornecedores = nw.tabela("suppliers")

produtos.head()

`info()` é o `\d tabela` do pandas: mostra colunas, tipos e quantos valores não nulos existem.
Repare que `pandas` já traz os tipos do PostgreSQL traduzidos — e que `int64` não aceita nulo,
o que explica por que colunas com `NULL` viram `float64` ou `object`.

In [ ]:
produtos.info()

---

## Dicionário de tradução



| SQL | pandas |
|---|---|
| `SELECT col1, col2` | `df[["col1", "col2"]]` |
| `WHERE cond` | `df[df["col"] > x]` ou `df.query("col > x")` |
| `WHERE col BETWEEN a AND b` | `df["col"].between(a, b)` |
| `WHERE col LIKE '%x%'` | `df["col"].str.contains("x")` |
| `WHERE col IN (...)` | `df["col"].isin([...])` |
| `WHERE col IS NULL` | `df["col"].isna()` |
| `ORDER BY col DESC` | `df.sort_values("col", ascending=False)` |
| `ORDER BY col DESC LIMIT n` | `df.nlargest(n, "col")` |
| `DISTINCT` | `df.drop_duplicates()` |
| `INNER JOIN` | `df.merge(outro, on="chave")` |
| `LEFT JOIN` | `df.merge(outro, on="chave", how="left")` |
| `FULL OUTER JOIN` | `df.merge(outro, on="chave", how="outer")` |
| `CROSS JOIN` | `df.merge(outro, how="cross")` |
| `GROUP BY ... COUNT/SUM/AVG` | `df.groupby("col").agg(...)` |
| `COUNT(DISTINCT col)` | `.nunique()` |
| `HAVING cond` | filtrar **depois** do `groupby` |
| `UNION ALL` | `pd.concat([df1, df2])` |
| `UNION` | `pd.concat([df1, df2]).drop_duplicates()` |
| `WITH nome AS (...)` | uma variável: `nome = ...` |
| `AVG(x) OVER (PARTITION BY g)` | `df.groupby("g")["x"].transform("mean")` |
| `RANK() OVER (PARTITION BY g ORDER BY x)` | `df.groupby("g")["x"].rank(method="min")` |
| `LAG(x)` / `LEAD(x)` | `df["x"].shift(1)` / `df["x"].shift(-1)` |
| `DATE_TRUNC('month', d)` | `df["d"].dt.to_period("M")` |
| `EXTRACT(YEAR FROM d)` | `df["d"].dt.year` |
| `COALESCE(a, b)` | `df["a"].fillna(df["b"])` |
| `CASE WHEN` | `np.select([cond1, cond2], [v1, v2], default=v3)` |
| `CAST(x AS NUMERIC)` | `df["x"].astype("float64")` |
| `NOT EXISTS` | `merge(..., how="left", indicator=True)` + `_merge == "left_only"` |


**OBS.:** uma CTE não é uma construção especial, mas é uma variável. Quando você escreve `WITH vendas AS (...)`, em pandas você escreve `vendas = ...`. Toda a fluência em
encadear CTEs já é fluência em encadear DataFrames.

---

## Parte 1 — Filtro, ordenação e projeção

### Exemplo 1 · `WHERE` e `ORDER BY`

Lista básica 01, exercício 05:

```sql
  SELECT p.product_id, p.product_name
    FROM products p
   WHERE p.unit_price > 50
ORDER BY p.unit_price DESC
```

In [ ]:
sql = '''
  SELECT p.product_id, p.product_name
    FROM products p
   WHERE p.unit_price > 50
ORDER BY p.unit_price DESC
'''

caros = (produtos
         .loc[produtos["unit_price"] > 50, ["product_id", "product_name", "unit_price"]]
         .sort_values("unit_price", ascending=False))

nw.conferir(sql, caros)
caros

`.loc[linhas, colunas]` faz o `WHERE` e o `SELECT` de uma vez. É o formato mais próximo do SQL.

A partir do **pandas 3.0** o *Copy-on-Write* é o comportamento padrão: toda seleção devolve uma
cópia lógica, então atribuir em um recorte nunca altera o DataFrame de origem em silêncio — a
armadilha que no pandas 2 produzia o aviso `SettingWithCopyWarning`. Se você encontrar material
antigo falando desse aviso, ele não se aplica mais ao ambiente deste repositório.

### Exemplo 2 · `BETWEEN`, `LIKE` e funções de texto

Exercícios 06, 09, 10 e 11 da mesma lista, todos de uma vez:

In [ ]:
entre_50_e_200 = produtos[produtos["unit_price"].between(50, 200)].sort_values("product_name")
contem_tofu    = produtos[produtos["product_name"].str.contains("Tofu")]
comeca_com_t   = produtos[produtos["product_name"].str.startswith("T")]
nome_com_4     = produtos[produtos["product_name"].str.len() == 4]

for nome, df in [("BETWEEN 50 AND 200", entre_50_e_200),
                 ("LIKE '%Tofu%'", contem_tofu),
                 ("LIKE 'T%'", comeca_com_t),
                 ("LENGTH(...) = 4", nome_com_4)]:
    print(f"{nome:<22} -> {len(df):>3} linhas")

O acessador `.str` expõe os métodos de texto do Python vetorizados — `contains`, `startswith`,
`len`, `upper`, `replace`. É onde vive todo o `LIKE` e as funções de string do SQL.

### Exemplo 3 · `ORDER BY ... LIMIT`

Lista básica 02, exercícios 38 e 39:

```sql
  SELECT product_name, CAST(unit_price AS NUMERIC(10,2)) AS UnitPrice
    FROM products
ORDER BY unit_price DESC
LIMIT 2
```

In [ ]:
produtos.nlargest(2, "unit_price")[["product_name", "unit_price"]]

`nlargest(n, col)` é mais direto e mais rápido que `sort_values().head(n)`, porque não precisa
ordenar a tabela inteira. Existe também `nsmallest`.

---

## Parte 2 — Junções

### Exemplo 4 · `INNER JOIN`

Lista básica 01, exercício 07:

```sql
SELECT p.product_id, p.product_name, c.category_id
  FROM products p
  JOIN categories c ON c.category_id = p.category_id
 WHERE c.category_id IN (2, 4, 6)
```

In [ ]:
sql = '''
SELECT p.product_id, p.product_name, c.category_id
  FROM products p
  JOIN categories c ON c.category_id = p.category_id
 WHERE c.category_id IN (2, 4, 6)
'''

selecionados = (produtos
                .merge(categorias, on="category_id")
                .query("category_id in [2, 4, 6]")
                [["product_id", "product_name", "category_id"]])

nw.conferir(sql, selecionados)
selecionados.head()

`merge` usa `how="inner"` por padrão, igual ao `JOIN` sem qualificador do SQL. Quando as colunas
de junção têm nomes diferentes, use `left_on=` e `right_on=`.

**Cuidado que o SQL não te dá:** se a chave tiver duplicatas dos dois lados, o `merge` multiplica
as linhas — exatamente como o `JOIN`, mas sem um plano de execução para te avisar. Confira
`len()` antes e depois de toda junção.

### Exemplo 5 · `LEFT JOIN` e auto-junção

Lista básica 02, exercício 36 — funcionário e seu chefe:

```sql
   SELECT e1.first_name, e1.last_name, e2.first_name, e2.last_name
     FROM employees e1
LEFT JOIN employees e2 ON e2.employee_id = e1.reports_to
```

In [ ]:
chefes = funcionarios[["employee_id", "first_name", "last_name"]]

hierarquia = (funcionarios
              .merge(chefes,
                     left_on="reports_to", right_on="employee_id",
                     how="left", suffixes=("", "_chefe"))
              [["first_name", "last_name", "first_name_chefe", "last_name_chefe"]])

hierarquia

`suffixes=("", "_chefe")` resolve a colisão de nomes que no SQL você resolveria com os apelidos
`e1` e `e2`. Deixar o primeiro sufixo vazio mantém as colunas da esquerda com o nome original.

### Exemplo 6 · Junções encadeadas com `DISTINCT`

Lista básica 02, exercício 35 — clientes e os produtos que compraram:

```sql
  SELECT DISTINCT c.company_name, p.product_name
    FROM customers     c
    JOIN orders        o  ON o.customer_id = c.customer_id
    JOIN order_details od ON od.order_id = o.order_id
    JOIN products      p  ON p.product_id = od.product_id
ORDER BY c.company_name, p.product_name
```

In [ ]:
sql = '''
  SELECT DISTINCT c.company_name, p.product_name
    FROM customers     c
    JOIN orders        o  ON o.customer_id = c.customer_id
    JOIN order_details od ON od.order_id = o.order_id
    JOIN products      p  ON p.product_id = od.product_id
ORDER BY c.company_name, p.product_name
'''

cliente_produto = (clientes
                   .merge(pedidos, on="customer_id")
                   .merge(itens, on="order_id")
                   .merge(produtos, on="product_id")
                   [["company_name", "product_name"]]
                   .drop_duplicates()
                   .sort_values(["company_name", "product_name"]))

nw.conferir(sql, cliente_produto)
cliente_produto.head()

O encadeamento de `merge` lê na mesma ordem do `FROM ... JOIN ... JOIN`. A diferença é que aqui a
ordem importa para o desempenho e você é quem decide — não há otimizador reordenando por você.

---

## Parte 3 — Agregação

### Exemplo 7 · `GROUP BY` com expressão calculada

Total de vendas por categoria. Note o padrão do Northwind: o valor da linha é
`quantity * unit_price * (1 - discount)`. Vale calcular essa coluna uma vez e reaproveitar.

In [ ]:
itens_valorizados = itens.assign(
    valor=itens["quantity"] * itens["unit_price"] * (1 - itens["discount"])
)

vendas_categoria = (itens_valorizados
                    .merge(produtos[["product_id", "category_id"]], on="product_id")
                    .merge(categorias[["category_id", "category_name"]], on="category_id")
                    .groupby(["category_id", "category_name"], as_index=False)
                    .agg(total_vendas=("valor", "sum"))
                    .sort_values("total_vendas", ascending=False))

vendas_categoria

Dois detalhes que economizam muito tempo depois:

- **`as_index=False`** mantém as colunas do agrupamento como colunas normais, em vez de virarem
  índice. É o comportamento que corresponde ao `GROUP BY` do SQL.
- **Agregação nomeada** — `agg(nome=("coluna", "função"))` — é o equivalente direto de
  `SUM(valor) AS total_vendas`. Prefira sempre a esta forma; ela evita colunas com nomes em
  múltiplos níveis, que são a maior fonte de confusão em pandas.

### Exemplo 8 · CTE com `HAVING` — o padrão mais importante

Lista avançada 05, exercício 01. Aqui está a tradução conceitual central da fase:

```sql
WITH vendas_cliente AS
(
      SELECT o.customer_id,
             COUNT(DISTINCT o.order_id)                           AS total_pedidos,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total_vendas
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT *
    FROM vendas_cliente
   WHERE total_pedidos > 10
ORDER BY total_vendas DESC;
```

In [ ]:
sql = '''
WITH vendas_cliente AS
(
      SELECT o.customer_id,
             COUNT(DISTINCT o.order_id)                           AS total_pedidos,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total_vendas
        FROM orders        o
        JOIN order_details od ON od.order_id = o.order_id
    GROUP BY o.customer_id
)
  SELECT *
    FROM vendas_cliente
   WHERE total_pedidos > 10
ORDER BY total_vendas DESC
'''

# A CTE vira simplesmente uma variável.
vendas_cliente = (pedidos
                  .merge(itens_valorizados, on="order_id")
                  .groupby("customer_id", as_index=False)
                  .agg(total_pedidos=("order_id", "nunique"),
                       total_vendas=("valor", "sum")))

# O HAVING vira um filtro comum, aplicado depois do agrupamento.
fieis = (vendas_cliente[vendas_cliente["total_pedidos"] > 10]
         .sort_values("total_vendas", ascending=False))

nw.conferir(sql, fieis)
fieis

Três equivalências neste exemplo:

| SQL | pandas |
|---|---|
| `WITH vendas_cliente AS (...)` | `vendas_cliente = ...` |
| `COUNT(DISTINCT o.order_id)` | `("order_id", "nunique")` |
| `HAVING` / `WHERE` sobre a CTE | filtro comum, **depois** do `groupby` |

A diferença prática entre `WHERE` e `HAVING` desaparece em pandas: como o agrupamento produz um
DataFrame novo, filtrar antes ou depois é só uma questão de em qual variável você aplica o filtro.

---

## Parte 4 — Subconsultas e anti-junções

### Exemplo 9 · Subconsulta correlacionada

Lista básica 02, exercício 40 — produtos acima do preço médio da própria categoria:

```sql
SELECT p.product_name, c.category_name, p.unit_price
  FROM products   p
  JOIN categories c ON c.category_id = p.category_id
 WHERE p.unit_price > (SELECT AVG(p2.unit_price)
                         FROM products p2
                        WHERE p2.category_id = p.category_id)
```

In [ ]:
acima_da_media = (produtos
                  .assign(media_categoria=produtos.groupby("category_id")["unit_price"]
                                                  .transform("mean"))
                  .query("unit_price > media_categoria")
                  .merge(categorias[["category_id", "category_name"]], on="category_id")
                  [["product_name", "category_name", "unit_price", "media_categoria"]]
                  .sort_values(["category_name", "unit_price"], ascending=[True, False]))

acima_da_media.head(10)

**`transform` é a peça que faltava.** Ele agrega por grupo mas devolve um valor **por linha**,
alinhado ao índice original — que é exatamente o que uma subconsulta correlacionada, ou uma
função de janela `AVG(...) OVER (PARTITION BY ...)`, faz.

Compare com `agg`, que devolve **uma linha por grupo**. É a distinção que mais confunde quem chega
do SQL, e você tem a vantagem de já ter o modelo mental certo: `agg` é `GROUP BY`, `transform` é
`OVER (PARTITION BY ...)`.

### Exemplo 10 · `NOT IN` e `NOT EXISTS`

Lista básica 02, exercício 37 — quem não é chefe de ninguém:

In [ ]:
ids_de_chefes = funcionarios["reports_to"].dropna().unique()
nao_sao_chefes = funcionarios[~funcionarios["employee_id"].isin(ids_de_chefes)]

print(f"{len(nao_sao_chefes)} funcionários não chefiam ninguém")
nao_sao_chefes[["employee_id", "first_name", "last_name"]]

Para o caso mais geral — o `NOT EXISTS` sobre uma junção — o padrão é a **anti-junção** com
`indicator=True`. Clientes que nunca fizeram pedido:

In [ ]:
sem_pedido = (clientes
              .merge(pedidos[["customer_id"]].drop_duplicates(),
                     on="customer_id", how="left", indicator=True)
              .query("_merge == 'left_only'")
              .drop(columns="_merge"))

print(f"{len(sem_pedido)} clientes sem nenhum pedido")
sem_pedido[["customer_id", "company_name", "country"]]

`indicator=True` cria a coluna `_merge` com os valores `both`, `left_only` e `right_only`. É a
forma mais legível de expressar `NOT EXISTS`, `EXISTS` e junções externas completas em pandas —
e o mais próximo que existe de um `FULL OUTER JOIN` com diagnóstico.

---

## Parte 5 — Funções de janela

### Exemplo 11 · `RANK() OVER (PARTITION BY ...)`

Os três produtos mais caros de cada categoria:

```sql
WITH ranqueados AS
(
    SELECT product_name, category_id, unit_price,
           RANK() OVER (PARTITION BY category_id ORDER BY unit_price DESC) AS posicao
      FROM products
)
SELECT * FROM ranqueados WHERE posicao <= 3
```

In [ ]:
ranqueados = produtos.assign(
    posicao=produtos.groupby("category_id")["unit_price"]
                    .rank(method="min", ascending=False)
)

top3 = (ranqueados[ranqueados["posicao"] <= 3]
        .merge(categorias[["category_id", "category_name"]], on="category_id")
        .sort_values(["category_name", "posicao"])
        [["category_name", "product_name", "unit_price", "posicao"]])

top3.head(12)

O parâmetro `method` escolhe qual função de janela do SQL você está reproduzindo:

| SQL | pandas |
|---|---|
| `RANK()` | `rank(method="min")` |
| `DENSE_RANK()` | `rank(method="dense")` |
| `ROW_NUMBER()` | `rank(method="first")` |

### Exemplo 12 · `LAG` e `DATE_TRUNC` — variação mensal

Padrão que aparece nas suas listas avançadas: faturamento por mês e a variação contra o mês anterior.

In [ ]:
vendas_mensais = (pedidos
                  .merge(itens_valorizados, on="order_id")
                  .assign(mes=lambda d: d["order_date"].dt.to_period("M"))
                  .groupby("mes", as_index=False)
                  .agg(faturamento=("valor", "sum"))
                  .sort_values("mes"))

vendas_mensais["mes_anterior"] = vendas_mensais["faturamento"].shift(1)
vendas_mensais["variacao_pct"] = (
    vendas_mensais["faturamento"] / vendas_mensais["mes_anterior"] - 1
) * 100

vendas_mensais.head(12)

`shift(1)` é o `LAG`, `shift(-1)` é o `LEAD`. **Ordene antes de usar** — ao contrário do SQL, onde
o `ORDER BY` faz parte da cláusula `OVER`, aqui a ordem das linhas é o que define o deslocamento.
Esquecer o `sort_values` é o erro mais comum, e ele não gera nenhum aviso.

Quando o deslocamento precisa respeitar um grupo — o `PARTITION BY` do `LAG` — encadeie o
`groupby`: `df.groupby("g")["x"].shift(1)`.

### Exemplo 13 · `CASE WHEN` e `COALESCE`

In [ ]:
faixas = np.select(
    [produtos["unit_price"] < 20,
     produtos["unit_price"] < 50],
    ["barato", "médio"],
    default="caro",
)

classificados = produtos.assign(
    faixa=faixas,
    fornecedor=produtos["supplier_id"].fillna(-1).astype("int64"),
)

classificados["faixa"].value_counts()

`np.select(condicoes, valores, default=...)` é o `CASE WHEN` — as condições são avaliadas em ordem
e a primeira verdadeira vence, exatamente como no SQL. Para um mapeamento simples de um para um,
`.map({...})` é mais legível.

`fillna` é o `COALESCE`. Repare que precisei do `astype` depois: em pandas, uma coluna inteira com
nulos vira `float64`, porque `int64` não representa `NaN`. É uma diferença real em relação ao
PostgreSQL e vale ter em mente ao gravar dados de volta no banco.

---

# Exercícios

A partir daqui é com você. As consultas abaixo saíram das suas próprias listas — o gabarito está
em [`estudos/sql/respostas/`](../../sql/respostas/).

Use `nw.conferir(sql, resultado)` em cada uma para validar contra o banco.

**Meta da fase: 30 consultas traduzidas.** As 13 dos exemplos já contam.

### Exercício 01

Produtos descontinuados, ordenados por nome.

```sql
SELECT product_id, product_name
  FROM products
 WHERE discontinued = 1
ORDER BY product_name
```

In [ ]:
sql = '''
SELECT product_id, product_name
  FROM products
 WHERE discontinued = 1
ORDER BY product_name
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 02

Quantidade de produtos por categoria, da maior para a menor.

```sql
  SELECT c.category_name, COUNT(*) AS total
    FROM products   p
    JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY total DESC
```

In [ ]:
sql = '''
  SELECT c.category_name, COUNT(*) AS total
    FROM products   p
    JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY total DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 03

Clientes do Brasil e da Argentina.

```sql
SELECT customer_id, company_name, country
  FROM customers
 WHERE country IN ('Brazil', 'Argentina')
```

In [ ]:
sql = '''
SELECT customer_id, company_name, country
  FROM customers
 WHERE country IN ('Brazil', 'Argentina')
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 04

Pedidos ainda não enviados.

```sql
SELECT order_id, customer_id, order_date
  FROM orders
 WHERE shipped_date IS NULL
```

In [ ]:
sql = '''
SELECT order_id, customer_id, order_date
  FROM orders
 WHERE shipped_date IS NULL
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 05

Ticket médio por funcionário. Requer juntar três tabelas e agregar.

```sql
  SELECT e.employee_id, e.first_name, e.last_name,
         AVG(od.quantity * od.unit_price * (1 - od.discount)) AS ticket_medio
    FROM employees     e
    JOIN orders        o  ON o.employee_id = e.employee_id
    JOIN order_details od ON od.order_id   = o.order_id
GROUP BY e.employee_id, e.first_name, e.last_name
ORDER BY ticket_medio DESC
```

In [ ]:
sql = '''
  SELECT e.employee_id, e.first_name, e.last_name,
         AVG(od.quantity * od.unit_price * (1 - od.discount)) AS ticket_medio
    FROM employees     e
    JOIN orders        o  ON o.employee_id = e.employee_id
    JOIN order_details od ON od.order_id   = o.order_id
GROUP BY e.employee_id, e.first_name, e.last_name
ORDER BY ticket_medio DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 06

Os 5 pedidos de maior valor. Lista avançada 05, exercício 02.

```sql
WITH total_pedido AS
(
      SELECT order_id, SUM(unit_price * quantity) AS total
        FROM order_details
    GROUP BY order_id
)
  SELECT *
    FROM total_pedido
ORDER BY total DESC
   LIMIT 5
```

In [ ]:
sql = '''
WITH total_pedido AS
(
      SELECT order_id, SUM(unit_price * quantity) AS total
        FROM order_details
    GROUP BY order_id
)
  SELECT *
    FROM total_pedido
ORDER BY total DESC
   LIMIT 5
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 07

Fornecedores com produtos em estoque baixo (menos de 10 unidades).

```sql
  SELECT s.company_name, COUNT(*) AS produtos_em_falta
    FROM suppliers s
    JOIN products  p ON p.supplier_id = s.supplier_id
   WHERE p.units_in_stock < 10
GROUP BY s.company_name
ORDER BY produtos_em_falta DESC
```

In [ ]:
sql = '''
  SELECT s.company_name, COUNT(*) AS produtos_em_falta
    FROM suppliers s
    JOIN products  p ON p.supplier_id = s.supplier_id
   WHERE p.units_in_stock < 10
GROUP BY s.company_name
ORDER BY produtos_em_falta DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 08

Frete médio por país de destino, apenas países acima da média global. Exige `transform`.

```sql
WITH frete_pais AS
(
      SELECT ship_country, AVG(freight) AS frete_medio
        FROM orders
    GROUP BY ship_country
)
SELECT *
  FROM frete_pais
 WHERE frete_medio > (SELECT AVG(freight) FROM orders)
```

In [ ]:
sql = '''
WITH frete_pais AS
(
      SELECT ship_country, AVG(freight) AS frete_medio
        FROM orders
    GROUP BY ship_country
)
SELECT *
  FROM frete_pais
 WHERE frete_medio > (SELECT AVG(freight) FROM orders)
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 09

Produtos que nunca foram vendidos. Anti-junção.

```sql
SELECT p.product_id, p.product_name
  FROM products p
 WHERE NOT EXISTS (SELECT 1
                     FROM order_details od
                    WHERE od.product_id = p.product_id)
```

In [ ]:
sql = '''
SELECT p.product_id, p.product_name
  FROM products p
 WHERE NOT EXISTS (SELECT 1
                     FROM order_details od
                    WHERE od.product_id = p.product_id)
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 10

Tempo médio de entrega em dias, por país. Cuidado com pedidos não enviados.

```sql
  SELECT ship_country,
         AVG(shipped_date - order_date) AS dias_medios
    FROM orders
   WHERE shipped_date IS NOT NULL
GROUP BY ship_country
ORDER BY dias_medios DESC
```

In [ ]:
sql = '''
  SELECT ship_country,
         AVG(shipped_date - order_date) AS dias_medios
    FROM orders
   WHERE shipped_date IS NOT NULL
GROUP BY ship_country
ORDER BY dias_medios DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 11

Faturamento por ano e trimestre.

```sql
  SELECT EXTRACT(YEAR    FROM o.order_date) AS ano,
         EXTRACT(QUARTER FROM o.order_date) AS trimestre,
         SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
    FROM orders        o
    JOIN order_details od ON od.order_id = o.order_id
GROUP BY 1, 2
ORDER BY 1, 2
```

In [ ]:
sql = '''
  SELECT EXTRACT(YEAR    FROM o.order_date) AS ano,
         EXTRACT(QUARTER FROM o.order_date) AS trimestre,
         SUM(od.quantity * od.unit_price * (1 - od.discount)) AS faturamento
    FROM orders        o
    JOIN order_details od ON od.order_id = o.order_id
GROUP BY 1, 2
ORDER BY 1, 2
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 12

Participação de cada produto no faturamento da sua categoria, em porcentagem. Exige `transform` sobre o resultado de um `groupby`.

```sql
WITH vendas AS
(
      SELECT p.category_id, p.product_name,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total
        FROM order_details od
        JOIN products      p ON p.product_id = od.product_id
    GROUP BY p.category_id, p.product_name
)
SELECT category_id, product_name, total,
       100.0 * total / SUM(total) OVER (PARTITION BY category_id) AS pct_categoria
  FROM vendas
ORDER BY category_id, pct_categoria DESC
```

In [ ]:
sql = '''
WITH vendas AS
(
      SELECT p.category_id, p.product_name,
             SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total
        FROM order_details od
        JOIN products      p ON p.product_id = od.product_id
    GROUP BY p.category_id, p.product_name
)
SELECT category_id, product_name, total,
       100.0 * total / SUM(total) OVER (PARTITION BY category_id) AS pct_categoria
  FROM vendas
ORDER BY category_id, pct_categoria DESC
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 13

Intervalo médio, em dias, entre pedidos consecutivos de cada cliente. Exige `shift` dentro de `groupby`.

```sql
WITH ordenados AS
(
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS anterior
      FROM orders
)
  SELECT customer_id, AVG(order_date - anterior) AS dias_entre_pedidos
    FROM ordenados
   WHERE anterior IS NOT NULL
GROUP BY customer_id
ORDER BY dias_entre_pedidos
```

In [ ]:
sql = '''
WITH ordenados AS
(
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS anterior
      FROM orders
)
  SELECT customer_id, AVG(order_date - anterior) AS dias_entre_pedidos
    FROM ordenados
   WHERE anterior IS NOT NULL
GROUP BY customer_id
ORDER BY dias_entre_pedidos
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 14

Clientes que compraram em todas as categorias de produto. Compare a contagem de categorias distintas por cliente com o total de categorias.

```sql
SELECT c.customer_id, c.company_name
  FROM customers c
  JOIN orders        o  ON o.customer_id = c.customer_id
  JOIN order_details od ON od.order_id   = o.order_id
  JOIN products      p  ON p.product_id  = od.product_id
GROUP BY c.customer_id, c.company_name
HAVING COUNT(DISTINCT p.category_id) = (SELECT COUNT(*) FROM categories)
```

In [ ]:
sql = '''
SELECT c.customer_id, c.company_name
  FROM customers c
  JOIN orders        o  ON o.customer_id = c.customer_id
  JOIN order_details od ON od.order_id   = o.order_id
  JOIN products      p  ON p.product_id  = od.product_id
GROUP BY c.customer_id, c.company_name
HAVING COUNT(DISTINCT p.category_id) = (SELECT COUNT(*) FROM categories)
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 15

Hierarquia completa de funcionários com o nível de cada um. Lista avançada 05, exercício 22 — a CTE recursiva. Em pandas, resolve-se com um laço que junta o resultado a si mesmo até não sobrar ninguém.

```sql
WITH RECURSIVE hierarquia AS
(
    SELECT employee_id, first_name, last_name, reports_to, 1 AS nivel
      FROM employees
     WHERE reports_to IS NULL

     UNION ALL

    SELECT e.employee_id, e.first_name, e.last_name, e.reports_to, h.nivel + 1
      FROM employees  e
      JOIN hierarquia h ON h.employee_id = e.reports_to
)
SELECT * FROM hierarquia ORDER BY nivel, last_name
```

In [ ]:
sql = '''
WITH RECURSIVE hierarquia AS
(
    SELECT employee_id, first_name, last_name, reports_to, 1 AS nivel
      FROM employees
     WHERE reports_to IS NULL

     UNION ALL

    SELECT e.employee_id, e.first_name, e.last_name, e.reports_to, h.nivel + 1
      FROM employees  e
      JOIN hierarquia h ON h.employee_id = e.reports_to
)
SELECT * FROM hierarquia ORDER BY nivel, last_name
'''

# resultado = ...

# nw.conferir(sql, resultado)

### Exercício 16

Escolha livre: pegue qualquer consulta da lista avançada 05 que você ainda não traduziu e faça a ponte.

```sql
-- sua escolha
```

In [ ]:
sql = '''
-- sua escolha
'''

# resultado = ...

# nw.conferir(sql, resultado)

---

## Fechamento da fase

Quando tiver 30 consultas traduzidas e conferindo, você terminou a Fase 1. O sinal de que a fase
pegou não é o número — é você conseguir olhar uma consulta e já enxergar o encadeamento de pandas
antes de escrever a primeira linha.

**Três coisas que vale registrar enquanto traduz**, porque vão virar conteúdo de entrevista:

1. **Onde o pandas ficou mais claro que o SQL** — normalmente em transformações encadeadas, onde
   cada passo fica em uma variável com nome.
2. **Onde o SQL ficou mais claro** — normalmente em junções múltiplas e em recursão.
3. **Onde os dois divergiram no resultado** — quase sempre por causa de `NULL` contra `NaN`, ou
   de tipo inteiro virando ponto flutuante. Essa é a diferença semântica de verdade entre as duas
   ferramentas, e saber explicá-la vale mais do que decorar a API.

**Próximo passo:** Fase 2 — análise exploratória e visualização, com o Projeto 1 em
[`projetos/`](../../../projetos/).